In [1]:
import pandas as pd
import numpy as np
import joblib

In [2]:
# Statistical dataset
final_df = pd.read_csv(
    "../data/processed/final_df.csv"
)

# Original dataset
raw_df = pd.read_csv(
    "../data/raw/counselling_system_Dataset.csv"
)

# ML model
model = joblib.load(
    "../backend/app/models/xgb_model.pkl"
)

# OHE encoder
ohe = joblib.load(
    "../backend/app/models/onehot_encoder.pkl"
)

print("✅ All files loaded")

✅ All files loaded


In [3]:
branch_strength = (

    raw_df
    .groupby("Branch")["ClosingRank"]
    .mean()
    .reset_index()
)

In [4]:
MAX_RANK = raw_df["ClosingRank"].max()

branch_strength["Competitiveness"] = (

    1 -

    (
        branch_strength["ClosingRank"] /
        MAX_RANK
    )
)

In [5]:
branch_strength = branch_strength[

    [
        "Branch",
        "Competitiveness"
    ]
]

branch_strength.head()

,Branch,Competitiveness
0,AI,0.712762
1,AIAIDS,0.802102
2,AIML,0.804945
3,AIR,0.856439
4,AUTO,0.723241


In [6]:
final_df = final_df.merge(

    branch_strength,

    on="Branch",

    how="left"
)

In [10]:
final_df.sample(5)

,Branch,Category,Round,Domicile,MinP,MaxP,MeanP,StdP,TotalSeats,BranchSeats,TotalCandidates,Trend,Competitiveness
109,AIML,OBC/X/OP,R1_Upgrade,Y,0.663541,0.880467,0.792906,0.083083,57.400000,15.200000,1.169627e+06,0.039272,0.804945
1007,ITAIAR,UR/X/F,INTERNAL,Y,0.747783,0.871725,0.818099,0.043850,49.333333,1.888889,1.254818e+06,0.018050,0.801605
956,IT,UR/S/F,INTERNAL,Y,0.602165,0.602165,0.602165,0.000000,107.000000,1.000000,1.475103e+06,0.000000,0.813498
851,ET,FW/OP,R2,Y,0.842855,0.935832,0.888659,0.046504,46.666667,1.333333,1.175812e+06,-0.012136,0.758907
455,CSD,OBC/X/F,R1,Y,0.846618,0.907661,0.872004,0.024115,91.200000,2.600000,1.169627e+06,-0.003452,0.801656


In [11]:
raw_df.head()

,Year,Round,Institute,Branch,Category,Domicile,OpeningRank,ClosingRank,BranchSeats,TotalSeats,TotalCandidates
0,2014.0,R1,MITS,BT,UR/X/OP,Y,78121.0,80003.0,2.0,30.0,1356000.0
1,2014.0,R1,MITS,BT,OBC/X/OP,Y,122826.0,133005.0,3.0,30.0,1356000.0
2,2014.0,R1,MITS,BT,UR/X/OP,Y,99915.0,137259.0,8.0,30.0,1356000.0
3,2014.0,R1,MITS,BT,UR/X/F,Y,113348.0,179038.0,4.0,30.0,1356000.0
4,2014.0,R1,MITS,BT,UR/NCC/OP,Y,190302.0,190302.0,1.0,30.0,1356000.0


In [12]:
model

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [13]:
ohe

,categories,'auto'
,drop,None
,sparse_output,False
,dtype,<class 'numpy.float64'>
,handle_unknown,'ignore'
,min_frequency,None
,max_categories,None
,feature_name_combiner,'concat'


In [14]:
current_year = raw_df["Year"].max()

# Take year from user
selected_year = int(input("Enter Year: "))

# Check if year exists
if selected_year in raw_df["Year"].values:

    CURRENT_TOTAL_CANDIDATES = int(
        raw_df.loc[
            raw_df["Year"] == selected_year,
            "TotalCandidates"
        ].iloc[0]
    )

    print("Selected Year:", selected_year)
    print("Candidates:", CURRENT_TOTAL_CANDIDATES)

else:
    print("Year not found in dataset")

Selected Year: 2024
Candidates: 1415110


In [16]:
user_rank = 84000
user_category = "OBC/X/OP"
user_round = "R1"
user_domicile = "Y"

In [18]:
user_percentile = (

    1 - (
        user_rank /
        CURRENT_TOTAL_CANDIDATES
    )
)

print("User Percentile:", user_percentile)

User Percentile: 0.9406406569100635


In [19]:
filtered_df = final_df[

    (final_df["Category"] == user_category) &
    (final_df["Round"] == user_round) &
    (final_df["Domicile"] == user_domicile)

].copy()

filtered_df.head()

,Branch,Category,Round,Domicile,MinP,MaxP,MeanP,StdP,TotalSeats,BranchSeats,TotalCandidates,Trend,Competitiveness
19,AI,OBC/X/OP,R1,Y,0.863111,0.892761,0.877936,0.020966,115.500000,10.500000,1.445106e+06,-0.029650,0.712762
58,AIAIDS,OBC/X/OP,R1,Y,0.829482,0.911058,0.882472,0.031965,75.600000,9.000000,1.169627e+06,0.013149,0.802102
108,AIML,OBC/X/OP,R1,Y,0.804610,0.905100,0.864162,0.047940,74.200000,12.000000,1.169627e+06,0.025005,0.804945
151,AIR,OBC/X/OP,R1,Y,0.900874,0.900874,0.900874,0.000000,69.000000,6.000000,1.023000e+06,0.000000,0.856439
185,AUTO,OBC/X/OP,R1,Y,0.156003,0.926221,0.688286,0.309072,59.888889,10.111111,1.088858e+06,-0.100000,0.723241


In [20]:
def calculate_probability(
    user_p,
    mean_p,
    std_p,
    trend
):

    adjusted_mean = mean_p + trend

    delta = user_p - adjusted_mean

    k = 25

    probability = (
        1 /
        (1 + np.exp(-k * delta))
    )

    probability *= (
        1 - min(std_p, 0.3)
    )

    return probability * 100

In [21]:
filtered_df["ExpectedClosingRank"] = (

    (1 - filtered_df["MeanP"]) *
    CURRENT_TOTAL_CANDIDATES
)

In [22]:
filtered_df["RankGap"] = (

    filtered_df["ExpectedClosingRank"] -
    user_rank
)

In [23]:
def calculate_gap_score(gap):

    if gap >= 120000:
        return 95

    elif gap >= 70000:
        return 85

    elif gap >= 30000:
        return 70

    elif gap >= 10000:
        return 55

    elif gap >= 0:
        return 40

    elif gap >= -10000:
        return 25

    else:
        return 10

In [32]:
results = []

for _, row in filtered_df.iterrows():

    # ---------------------------------
    # Statistical Probability
    # ---------------------------------

    stat_prob = calculate_probability(

        user_percentile,

        row["MeanP"],
        row["StdP"],
        row["Trend"]
    )

    # ---------------------------------
    # Rank Gap Score
    # ---------------------------------

    gap_score = calculate_gap_score(
        row["RankGap"]
    )

    # ---------------------------------
    # Statistical Fusion
    # ---------------------------------

    statistical_probability = (

        0.7 * stat_prob +
        0.3 * gap_score
    )

    # ---------------------------------
    # ML INPUT
    # ---------------------------------

    ml_input = pd.DataFrame([{

        "Year": current_year,

        "Rank": user_rank,

        "TotalCandidates":
            CURRENT_TOTAL_CANDIDATES,

        "TotalSeats":
            row["TotalSeats"],

        "BranchSeats":
            row["BranchSeats"],

        "RankPercentile":
            user_percentile,

        "SeatShare":
            row["BranchSeats"] /
            row["TotalSeats"],

        "Branch":
            row["Branch"],

        "Category":
            user_category,

        "Round":
            user_round,

        "Domicile":
            user_domicile
    }])

    # ---------------------------------
    # OHE TRANSFORM
    # ---------------------------------

    encoded_array = ohe.transform(

        ml_input[
            [
                "Branch",
                "Category",
                "Round",
                "Domicile"
            ]
        ]
    )

    encoded_df = pd.DataFrame(

        encoded_array,

        columns=ohe.get_feature_names_out(
            [
                "Branch",
                "Category",
                "Round",
                "Domicile"
            ]
        )
    )

    ml_input = ml_input.drop(
        columns=[
            "Branch",
            "Category",
            "Round",
            "Domicile"
        ]
    )

    ml_input = pd.concat(
        [
            ml_input.reset_index(drop=True),
            encoded_df.reset_index(drop=True)
        ],
        axis=1
    )

    # ---------------------------------
    # ALIGN COLUMNS
    # ---------------------------------

    model_features = model.feature_names_in_

    ml_input = ml_input.reindex(
        columns=model_features,
        fill_value=0
    )

    # ---------------------------------
    # ML PROBABILITY
    # ---------------------------------

    ml_probability = (

        model.predict_proba(
            ml_input
        )[0][1]
    ) * 100

    # ---------------------------------
    # FINAL HYBRID FUSION
    # ---------------------------------

    final_probability = (

        0.5 * statistical_probability +
        0.5 * ml_probability
    )

    # ---------------------------------
    # COMPETITIVENESS SCORE
    # ---------------------------------

    competitive_score = (
        row["Competitiveness"] * 100
    )

    # ---------------------------------
    # FINAL COUNSELLING SCORE
    # ---------------------------------

    final_counselling_score = (

        0.5 * final_probability +

        0.4 * competitive_score
    )

    results.append({

    "Branch":
        row["Branch"],

    "StatisticalProb":
        round(statistical_probability, 2),

    "MLProb":
        round(ml_probability, 2),

    "FinalProbability":
        round(final_probability, 2),

    "Competitiveness":
        round(competitive_score, 2),

    "CounsellingScore":
        round(final_counselling_score, 2),

    "ExpectedClosingRank":
        int(row["ExpectedClosingRank"]),

    "RankGap":
        int(row["RankGap"])
})

results_df = pd.DataFrame(results)

In [33]:
def classify(prob):

    if prob >= 85:
        return "Very Safe"

    elif prob >= 70:
        return "Safe"

    elif prob >= 50:
        return "Moderate"

    elif prob >= 30:
        return "Risky"

    else:
        return "Dream"

In [34]:
results_df["Status"] = (

    results_df["FinalProbability"]
    .apply(classify)
)

In [35]:
results_df = results_df.sort_values(

    by=[
        
        "FinalProbability"
    ],

    ascending= False
)

results_df.head(40)

,Branch,StatisticalProb,MLProb,FinalProbability,Competitiveness,CounsellingScore,ExpectedClosingRank,RankGap,Status
12,EACE,98.47,100.000000,99.24,59.48,73.41,530981,446981,Very Safe
8,CSBS,93.42,99.980003,96.70,71.89,77.11,252877,168877,Very Safe
5,BT,92.83,99.980003,96.41,74.39,77.96,234621,150621,Very Safe
18,ITAIAR,90.96,99.980003,95.47,80.16,79.80,241674,157674,Very Safe
20,MAC,90.36,100.000000,95.18,72.57,76.62,541384,457384,Very Safe
16,ET,89.56,99.970001,94.77,75.89,77.74,220153,136153,Very Safe
19,ITIOT,88.37,99.989998,94.18,74.50,76.89,368108,284108,Very Safe
0,AI,87.84,99.709999,93.77,71.28,75.40,172733,88733,Very Safe
15,EEIOT,84.93,100.000000,92.46,69.25,73.93,964805,880805,Very Safe
14,EE,80.95,99.989998,90.47,70.28,73.35,362544,278544,Very Safe
